# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con facebook/bart-large-mnli

Model page: https://huggingface.co/facebook/bart-large-mnli

Entailment is calculated in both directions (forward probs (Arg1->Arg2) and backward probs (Arg2->Arg1)), then the label is decided with the following criteria:
   
Bidirectional decision:
  - Rephrase: high entailment both ways, low contradiction
  - Attack: contradiction high either way
  - Support: entailment high at least one way, and not clearly neutral/contradictory
  - No Relationship: otherwise

#Keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships Keywords"

model_name = "facebook/bart-large-mnli"
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 50                 
MODEL_TAG = "bart"      
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# figure out label 
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

print(f'DeBERTa v3 base mnli labels {label2id}')

IDX_ENT = label2id["entailment"]
IDX_CON = label2id["contradiction"]
IDX_NEU = label2id["neutral"]

# Heuristic thresholds 
ENT_THR       = 0.50   
CONTR_THR     = 0.50   
REPHRASE_THR  = 0.75  
MAX_NEUTRAL   = 0.70   

# Margins: “X wins by at least this much”
SUPPORT_MARGIN = 0.10  # p_ent - p_con must exceed this (either direction)
ATTACK_MARGIN  = 0.10  # p_con - p_ent must exceed this (either direction)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2025-08-25 11:45:28.012309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756122328.330379      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756122328.424540      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

DeBERTa v3 base mnli labels {'contradiction': 0, 'neutral': 1, 'entailment': 2}


In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2459, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 2459 (delta 9), reused 6 (delta 5), pack-reused 2444 (from 2)
Receiving objects: 100% (2459/2459), 97.50 MiB | 19.03 MiB/s, done.
Resolving deltas: 100% (2065/2065), done.
Updating files: 100% (1061/1061), done.


In [3]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def nli_probs(pairs, max_length: int):
    if not pairs:
        return np.zeros((0,3)), np.zeros((0,3))

    a1 = [preprocess_text(x[0]) for x in pairs]
    a2 = [preprocess_text(x[1]) for x in pairs]

    # forward: premise=a1, hypothesis=a2
    enc_f = tokenizer(a1, a2, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_f = model(**enc_f).logits
    prob_f = torch.softmax(logits_f, dim=-1).detach().cpu().numpy()

    # backward: premise=a2, hypothesis=a1
    enc_b = tokenizer(a2, a1, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_b = model(**enc_b).logits
    prob_b = torch.softmax(logits_b, dim=-1).detach().cpu().numpy()

    return prob_f, prob_b

def decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b):
    # 1) Hard guard: if both sides look very neutral, say NR
    if max(p_neu_f, p_neu_b) >= MAX_NEUTRAL:
        return "No Relationship"

    # 2) Rephrase: bi-directional entailment and low contradiction
    if min(p_ent_f, p_ent_b) >= REPHRASE_THR and max(p_con_f, p_con_b) <= (1 - REPHRASE_THR):
        return "Rephrase"

    # 3) Attack: contradiction wins with margin OR strong contradiction
    if (
        (p_con_f - p_ent_f) >= ATTACK_MARGIN or
        (p_con_b - p_ent_b) >= ATTACK_MARGIN or
        p_con_f >= CONTR_THR or
        p_con_b >= CONTR_THR
    ):
        return "Attack"

    # 4) Support: entailment wins with margin OR clears lowered threshold
    if (
        (p_ent_f - p_con_f) >= SUPPORT_MARGIN or
        (p_ent_b - p_con_b) >= SUPPORT_MARGIN or
        p_ent_f >= ENT_THR or
        p_ent_b >= ENT_THR
    ):
        return "Support"

    # 5) Fallback
    return "No Relationship"


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_max_length(input_dir, prefix_substring, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            ids = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    if max_len == 0:
        max_len = 128
    return min(max_len, safety_limit)

def classify_relationships_deberta(input_dir, prefix_substring, model_tag=MODEL_TAG):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    MAX_LENGTH = compute_max_length(input_dir, prefix_substring, safety_limit=512)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    rel_col = f"rel_{model_tag}"

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                prob_f, prob_b = nli_probs(pairs, max_length=MAX_LENGTH)
                labels = []
                for i in range(len(pairs)):
                    pf = prob_f[i]; pb = prob_b[i]
                    p_ent_f, p_neu_f, p_con_f = pf[IDX_ENT], pf[IDX_NEU], pf[IDX_CON]
                    p_ent_b, p_neu_b, p_con_b = pb[IDX_ENT], pb[IDX_NEU], pb[IDX_CON]
                    lab = decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b)
                    labels.append(lab)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for _a1, _a2, _lab in first_examples:
                        print(f"- Arg1: {_a1}\n- Arg2: {_a2}\n  Label: {_lab}\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 100 < BATCH_SIZE:
                print(f"  Progress: {total_done}/{n} relations classified...")

        # Final label validation 
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
    return df


## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 140

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: All UN Memb

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
454,The SDGs have a significant impact on public m...,"build effective, accountable and inclusive ins...",16_4,16_7,NaN,Support,No Relationship,No Relationship
246,"At the midpoint of the 2030 Agenda, all countr...",SDSN puts a great emphasis on long-term nation...,0_12,0_13,NaN,Support,Support,No Relationship
268,SDSN puts a great emphasis on long-term nation...,While comparable country-level data are not ye...,0_13,0_22,NaN,No Relationship,No Relationship,No Relationship
304,"First, that UN Member States, at the 2023 SDG ...",The SDG Index is a flagship instrument to prom...,0_16,0_25,NaN,Support,Support,No Relationship
17,"At their core, the SDGs are an investment agen...",The SDGs require long-term directed change and...,0_0,0_18,NaN,Support,Support,No Relationship


rel_bart
No Relationship    495
Attack              10
Rephrase             2
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 148

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Increased funding from the multilateral development banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs;
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: Many of them lack an adequately high SDG commitment, and almost all lack access to the necessary financial means to implement the SDGs.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
563,The contribution of the SDGs towards a univers...,"The United States, as the world’s biggest econ...",0_26,7_1,NaN,Support,Support,No Relationship
1830,The contribution of the SDGs towards a univers...,Reform current institutional frameworks and de...,0_26,17_4,NaN,Support,Support,No Relationship
2963,The transition by 2050 of energy systems to ne...,The EU Green Deal has great potential to bring...,7_0,17_8,NaN,No Relationship,Support,No Relationship
3491,Almost all lack access to the necessary financ...,SDSN recommends an SDG Stimulus plan to close ...,10_2,17_1,NaN,Support,Support,No Relationship
1248,"At the global level, averaging across countrie...","combat desertification, and halt and reverse l...",0_2,15_0,NaN,Support,No Relationship,No Relationship


rel_bart
No Relationship    3821
Attack              114
Rephrase             13
Support              10
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 165

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The St

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
3344,Achieving the SDGs requires global cooperation...,"Climate change, peace, cybersecurity, reliable...",17_30,17_31,NaN,Support,Rephrase,No Relationship
784,"At their core, the SDGs are an investment agenda.",Achieving the SDGs will require a transformati...,0_19,0_25,NaN,Support,Support,No Relationship
432,"All UN Member States should present, at regula...",Long-term investment plans are essential for n...,0_9,0_28,NaN,Support,Support,No Relationship
700,The report’s recommendations include calling f...,As the aims of the 2030 Agenda are ever-evolvi...,0_16,0_37,NaN,Support,Support,No Relationship
295,All UN Member States and UN agencies can count...,"Despite this ominous news, the SDGs are still ...",0_6,0_17,NaN,Support,Support,No Relationship


rel_bart
No Relationship    3258
Attack               92
Rephrase              3
Support               1
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 178

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions;
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: to give all people the skills and knowledge to end poverty, protect the environment, and build peaceful and inclusive societies.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Although all governments are in principle committed to economic justice as enshrined in the Universal Declaration of Human Rights, and to the SDG ten

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
25941,To achieve the SDGs the world must both alter ...,the destruction of fisheries through over-fish...,8_0,14_1,NaN,No Relationship,No Relationship,No Relationship
24313,Private capital markets continue to direct lar...,The climate and biodiversity crises are driven...,7_2,13_21,NaN,No Relationship,Support,No Relationship
30867,"At this midpoint of the 2030 Agenda, all count...",Reform current institutional frameworks and de...,10_9,17_18,NaN,Support,Support,No Relationship
939,Long-term investment plans are essential for n...,The 2021 UN Food Systems Summit raised many ur...,0_28,2_1,NaN,Support,Rephrase,No Relationship
16547,Today’s land-use practices and food systems ha...,"Second, developed countries are not being held...",2_5,17_15,NaN,No Relationship,No Relationship,No Relationship


rel_bart
No Relationship    35464
Attack              1299
Rephrase              59
Support               10
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 234

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the SDGs are seriously off track
- Arg2: the SDGs are still achievable
  Label: Attack

- Arg1: the SDGs are seriously off track
- Arg2: it is critical that UN Member States adopt and implement the SDG Stimulus
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight
  Label: No Relationship

  Progress:

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
4087,The world would have escaped the current tumul...,scores are on average lowest for the second on...,1_20,1_28,NaN,No Relationship,No Relationship,No Relationship
1772,the world must devote an increased portion of ...,Each of these challenges requires large-scale ...,0_24,0_57,NaN,Support,Rephrase,No Relationship
4758,actions by governments at all levels to ensure...,"The SDSN’s flagship educational initiative, th...",4_2,4_8,NaN,Support,Support,No Relationship
5869,"Those related to hunger, sustainable diets, an...","The Convention on Biological Diversity, adopte...",15_7,15_13,NaN,No Relationship,Attack,No Relationship
6207,public policies must be pursued at all levels:...,SDG 16 recognizes the vital role of peaceful a...,16_11,16_15,NaN,Support,No Relationship,No Relationship


rel_bart
No Relationship    6653
Attack              308
Support               8
Rephrase              7
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 261

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: the SDGs are seriously off track
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises;
  Label: No Relationship

- Arg1: the SDGs are seriously off track
- Arg2: investing in statistical capacity, science, and data literacy are important priorities for achieving the SDGs
  Label: No Relationship


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
46165,Governments are only now mapping out pathways ...,"Four basic failures stand out: First, implemen...",3_5,16_9,NaN,No Relationship,No Relationship,No Relationship
67586,"At this rate, the likelihood of overshooting 1...","China has reiterated its support for the SDGs,...",13_3,17_7,NaN,Support,No Relationship,No Relationship
1869,The SDGs have a significant impact on public m...,achieving the SDGs,0_51,1_33,NaN,No Relationship,Rephrase,No Relationship
32754,"Education builds human capital, which in turn ...","At this rate, the likelihood of overshooting 1...",1_26,13_3,NaN,No Relationship,No Relationship,No Relationship
45400,The COVID-19 pandemic also severely depleted t...,negative scores for rich countries on SDGs 12–...,3_3,13_17,NaN,No Relationship,Support,No Relationship


rel_bart
No Relationship    67451
Attack              3617
Rephrase             123
Support               43
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 230

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
1794,"Yes, the world is off-track, but that is all t...",Achieving the SDGs will require a transformati...,0_30,0_40,NaN,Support,Support,No Relationship
3346,5. Sustainable cities: urban infrastructure an...,Governments must take the lead in all six area...,9_2,9_18,NaN,Support,Support,No Relationship
4070,Current Nationally Determined Contribution (ND...,The SDSN’s initiative contributed to the conce...,13_6,13_23,NaN,Support,Support,No Relationship
2430,Local governments have the front-line responsi...,The SDG Index and Dashboards track the annual ...,0_48,0_55,NaN,Support,Support,No Relationship
569,Achieving the SDGs requires global cooperation...,"Most urgently, UN Member States should adopt a...",0_8,0_14,NaN,Support,Support,No Relationship


rel_bart
No Relationship    5834
Attack              110
Support               4
Rephrase              3
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 297

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Most of the low-income and lower-middle income countries, home to more than the half of humanity, face major challenges in achieving most of the SDGs by 2030.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDG

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
22308,"According to IMF estimates in 2019, the financ...",The consequences of inaction in the face of th...,1_7,9_4,NaN,Attack,Support,No Relationship
34971,These should be complemented by anti-discrimin...,cities are where the climate battle will large...,5_3,11_12,NaN,No Relationship,No Relationship,No Relationship
10238,Although all governments are in principle comm...,Sustainable cities: urban infrastructure and s...,0_26,11_0,NaN,Support,No Relationship,No Relationship
45351,6. Transformation to universal digital access ...,SDSN calls on all nations to renounce violence...,9_12,17_10,NaN,Support,No Relationship,Attack
24171,"According to IMF estimates in 2019, the financ...",Open sharing of data and knowledge across thes...,1_7,17_28,NaN,Support,No Relationship,No Relationship


rel_bart
No Relationship    56591
Attack              1624
Rephrase              75
Support               20
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 239

Processing: /kaggle/working/TFM/Data/Relationships Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: All countries, poorer and richer alike, should use the half-way momentum to self-critically review and revise thei

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
5073,Biodiversity targets (SDG 15 and targets agree...,The scientific evidence points to global risks...,15_2,15_7,NaN,Support,Rephrase,No Relationship
1648,The SDGs are seriously off track (Figure 1.1)....,International financing flows should be aligne...,0_22,0_76,NaN,Support,Support,No Relationship
5195,The European Union – the world’s second-larges...,Private capital markets continue to direct lar...,15_11,15_12,NaN,Attack,Support,No Relationship
4983,Private capital markets continue to direct lar...,More ambitious policies and actions on climate...,13_14,13_23,NaN,Support,Support,No Relationship
756,All UN Member States and UN agencies can count...,The World Bank and the other MDBs should put t...,0_9,0_55,NaN,Support,Support,No Relationship


rel_bart
No Relationship    6402
Attack              127
Rephrase              5
Support               3
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 256

Processing: /kaggle/working/TFM/Data/Relationships Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 1. Increased funding from the multilateral develop-ment banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to investments in the SDGs;
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are serio

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_bart
2840,"Moreover, societal polarization, populism, and...",The consequences of inaction in the face of th...,0_18,3_2,NaN,Support,Support,No Relationship
32769,Some countries specifically refer to the SDGs ...,Governments are only now establishing R&D fund...,3_8,9_10,NaN,No Relationship,No Relationship,No Relationship
35862,Another is the Gateways to Public Digital Lear...,1. Universal quality education and innovation-...,4_9,9_0,NaN,Support,Support,No Relationship
52465,"Deep, chronic, and crippling under-investment ...","In that capacity, the SDSN is also supporting ...",10_12,17_24,NaN,Support,No Relationship,No Relationship
43265,Many countries continue to provide substantial...,National governments must ensure both the dome...,7_8,16_9,NaN,Support,Support,No Relationship


rel_bart
No Relationship    59088
Attack              1805
Rephrase              77
Support               21
Name: count, dtype: int64